# 🏌️ Mini Caddie — ONNX to Hailo HEF Compilation
Compile your trained YOLOv8 ONNX model to Hailo HEF format for deployment on Pi 5 + Hailo-8L

## Before you start
1. Download the Hailo Dataflow Compiler (DFC) wheel from https://hailo.ai/developer-zone/
   - You need a free Hailo Developer Zone account
   - Download the `.whl` file for Python 3.10 (Linux x86_64)
   - Upload it to your Google Drive in the same folder as your ONNX model
2. Your ONNX model (`mini_caddie_golf_best.onnx`) should already be in Google Drive from training
3. Set Colab runtime to **CPU** (no GPU needed for compilation)

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2: Install Hailo Dataflow Compiler
# You need to have downloaded the DFC .whl file from https://hailo.ai/developer-zone/
# and uploaded it to your Google Drive

import os

# Find the DFC wheel in your Drive
drive_path = '/content/drive/MyDrive'
whl_files = [f for f in os.listdir(drive_path) if f.endswith('.whl') and 'hailo' in f.lower()]

if whl_files:
    whl_path = os.path.join(drive_path, whl_files[0])
    print(f'Found DFC wheel: {whl_files[0]}')
    !pip install "{whl_path}" -q
    print('DFC installed!')
else:
    print('❌ No Hailo .whl file found in Google Drive!')
    print('Download from https://hailo.ai/developer-zone/ and upload to Drive.')
    print('Expected filename like: hailo_dataflow_compiler-3.x.x-py3-none-linux_x86_64.whl')

In [ ]:
# STEP 3: Unzip dataset for calibration images
# Hailo needs a set of images to calibrate the model during compilation
import zipfile, os

zip_path = '/content/drive/MyDrive/unified-golf-dataset.zip'
extract_path = '/content/dataset'

if os.path.exists(zip_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('Dataset extracted for calibration!')
else:
    print('Dataset zip not found — upload unified-golf-dataset.zip to Drive')

In [ ]:
# STEP 4: Copy ONNX model to working directory
import shutil, os

onnx_src = '/content/drive/MyDrive/mini_caddie_golf_best.onnx'
onnx_dst = '/content/mini_caddie_golf_best.onnx'

if os.path.exists(onnx_src):
    shutil.copy(onnx_src, onnx_dst)
    print('ONNX model copied!')
    print(f'Size: {os.path.getsize(onnx_dst) / 1024 / 1024:.1f} MB')
else:
    print('❌ ONNX model not found in Drive!')
    print('Make sure mini_caddie_golf_best.onnx is in your Drive root.')

In [ ]:
# STEP 5: Compile ONNX to HEF
# This is the main compilation step. It calibrates the model using
# your validation images and produces the Hailo Executable Format (.hef)

import subprocess, os

# Build calibration image list (use validation images)
val_images_dir = '/content/dataset/unified/valid/images'
calib_images = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)
                if f.endswith(('.jpg', '.png', '.jpeg'))][:100]

# Write calibration list to file
with open('/content/calib_images.txt', 'w') as f:
    f.write('\n'.join(calib_images))

print(f'Using {len(calib_images)} images for calibration')

# Compile using hailomz (Hailo Model Compiler)
# Hailo-8L architecture (for the HAT+ on Pi 5)
# 8 classes (your golf classes) + 1 background = 9 total after Hailo adds background
cmd = [
    'hailomz', 'compile',
    '--ckpt', '/content/mini_caddie_golf_best.onnx',
    '--calib-path', '/content/calib_images.txt',
    '--hw-arch', 'hailo8l',
    '--classes', '8',
    '--performance',
    '--output', '/content/mini_caddie_golf.hef'
]

print('Compiling... this takes 5-15 minutes')
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)

if os.path.exists('/content/mini_caddie_golf.hef'):
    print('✅ HEF compiled!')
    print(f'Size: {os.path.getsize("/content/mini_caddie_golf.hef") / 1024 / 1024:.1f} MB')
else:
    print('❌ HEF not found — check errors above')

In [ ]:
# STEP 6: Save HEF to Google Drive
import shutil, os

hef_src = '/content/mini_caddie_golf.hef'
hef_dst = '/content/drive/MyDrive/mini_caddie_golf.hef'

if os.path.exists(hef_src):
    shutil.copy(hef_src, hef_dst)
    print('✅ HEF saved to Google Drive!')
else:
    print('❌ HEF file not found — did compilation succeed?')

In [ ]:
# STEP 7: Create labels.json for deployment
# Hailo adds a background class at index 0, so all class IDs shift by +1

import json

labels = {
    "0": "background",
    "1": "golf_ball",
    "2": "golf_club",
    "3": "golf_club_head",
    "4": "golf_hole",
    "5": "golf_mat",
    "6": "person",
    "7": "player_not_ready",
    "8": "player_ready"
}

labels_path = '/content/drive/MyDrive/mini_caddie_labels.json'
with open(labels_path, 'w') as f:
    json.dump(labels, f, indent=2)

print('✅ labels.json saved to Drive!')
print(json.dumps(labels, indent=2))

## ✅ After compilation completes
Your Google Drive should now have:
- `mini_caddie_golf.hef` — the compiled Hailo model
- `mini_caddie_labels.json` — class labels (with background at 0)
- `mini_caddie_golf_best.onnx` — original ONNX (backup)
- `mini_caddie_golf_best.pt` — PyTorch weights (backup)

## Next: Deploy on Pi
1. Download `mini_caddie_golf.hef` and `mini_caddie_labels.json` from Drive
2. Copy to Pi (via GitHub or USB)
3. Run: `hailo-detect --hef-path mini_caddie_golf.hef --labels-json mini_caddie_labels.json --input rpi`